# 04 — YouTube-VIS Preprocessing

Goal: Extract per-frame annotations from YouTube-VIS for use in
SAM-2 video tracking experiments. Convert instance segmentation
annotations to per-frame YOLO detection labels. Build a
video_index.json for fast video-level access.

Inputs  : Youtube VIS/train/instances.json + JPEGImages/
          Youtube VIS/valid/instances.json + JPEGImages/
Outputs :
  - processed/youtube_vis/labels/{train,val}/<video_id>/<frame>.txt
  - processed/youtube_vis/video_index.json
  - processed/youtube_vis/ytvis_stats.csv
  - processed/youtube_vis/category_map.json

YouTube-VIS annotation format:
  - Videos contain N frames (JPEGImages/<video_id>/<frame_id>.jpg)
  - Each annotation has:
      "bboxes"     : list of [x,y,w,h] per frame (None if object absent)
      "segmentations": per-frame RLE or polygon masks
      "category_id": class label
      "video_id"   : which video

In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 1: Imports and config load
# ─────────────────────────────────────────────────────────────

import json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm
import cv2
import yaml

# ── Load config ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
CONFIG_PATH = "/content/drive/MyDrive/DLCV_OV_Analytics/configs/config.json"
with open(CONFIG_PATH) as f:
    cfg = json.load(f)

RAW_YTVIS  = cfg["raw_ytvis"]
PROC_YTVIS = cfg["proc_ytvis"]
METRICS_DIR = cfg["metrics_dir"]
VIZ_DIR    = cfg["viz_dir"]

# ── Output directories ────────────────────────────────────────
PROC_YV_LBL_TRAIN = os.path.join(PROC_YTVIS, "labels", "train")
PROC_YV_LBL_VAL   = os.path.join(PROC_YTVIS, "labels", "val")

for d in [PROC_YV_LBL_TRAIN, PROC_YV_LBL_VAL,
          os.path.join(PROC_YTVIS, "images", "train"),
          os.path.join(PROC_YTVIS, "images", "val")]:
    os.makedirs(d, exist_ok=True)

print("✅ Setup complete.")
print(f"   RAW  : {RAW_YTVIS}")
print(f"   PROC : {PROC_YTVIS}")

Mounted at /content/drive
✅ Setup complete.
   RAW  : /content/drive/MyDrive/DLCV_OV_Analytics/datasets/raw/Youtube VIS
   PROC : /content/drive/MyDrive/DLCV_OV_Analytics/datasets/processed/youtube_vis


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 2: Load instances.json for train and valid splits
# ─────────────────────────────────────────────────────────────

def load_ytvis_json(split):
    """Load YouTube-VIS instances.json for a given split."""
    json_path = os.path.join(RAW_YTVIS, split, "instances.json")
    if not os.path.exists(json_path):
        print(f"  ❌ Not found: {json_path}")
        return None, None   # FIX: return tuple, not None

    size_mb = os.path.getsize(json_path) / 1e6
    print(f"  Loading {split}/instances.json ({size_mb:.1f} MB)...")

    with open(json_path, "r") as f:
        data = json.load(f)

    n_videos   = len(data.get("videos", []))
    n_anns     = len(data.get("annotations", []))
    n_cats     = len(data.get("categories", []))
    categories = {c["id"]: c["name"] for c in data.get("categories", [])}

    print(f"    Videos      : {n_videos}")
    print(f"    Annotations : {n_anns}"
          + (" (withheld — expected)" if n_anns == 0 else ""))
    print(f"    Categories  : {n_cats}")
    print(f"    Cat names   : {list(categories.values())[:10]}...")

    return data, categories


print("=" * 50)
print("  YouTube-VIS — Loading Annotation JSONs")
print("=" * 50)

print("\nTrain:")
train_data, train_cats = load_ytvis_json("train")

print("\nValid:")
val_data, val_cats = load_ytvis_json("valid")

# Use train categories as master map
YTVIS_CATS = train_cats if train_cats else val_cats
print(f"\n✅ Master category map: {len(YTVIS_CATS)} classes")

  YouTube-VIS — Loading Annotation JSONs

Train:
  Loading train/instances.json (648.8 MB)...
    Videos      : 2985
    Annotations : 6283
    Categories  : 40
    Cat names   : ['airplane', 'bear', 'bird', 'boat', 'car', 'cat', 'cow', 'deer', 'dog', 'duck']...

Valid:
  Loading valid/instances.json (0.5 MB)...
    Videos      : 492
    Annotations : 0 (withheld — expected)
    Categories  : 40
    Cat names   : ['airplane', 'bear', 'bird', 'boat', 'car', 'cat', 'cow', 'deer', 'dog', 'duck']...

✅ Master category map: 40 classes


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 3: Build video index and save to Drive with cp && sync
# ─────────────────────────────────────────────────────────────
import shutil, subprocess

def cp_sync(src, dst_dir):
    result = subprocess.run(f"cp '{src}' '{dst_dir}/' && sync",
                            shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        raise IOError(f"cp failed: {result.stderr}")
    return os.path.join(dst_dir, os.path.basename(src))

def build_video_index(data, split):
    index    = {}
    img_root = os.path.join(RAW_YTVIS, split, "JPEGImages")
    if not os.path.isdir(img_root):
        print(f"  ⚠️  JPEGImages not found at: {img_root}")

    for video in data.get("videos", []):
        vid_id    = video["id"]
        filenames = video.get("file_names", [])
        vid_folder   = Path(filenames[0]).parent.name if filenames else str(vid_id)
        vid_img_path = os.path.join(img_root, vid_folder)
        index[vid_id] = {
            "split": split, "length": video["length"],
            "file_names": filenames, "vid_folder": vid_folder,
            "img_root": img_root, "vid_img_path": vid_img_path,
            "exists": os.path.isdir(vid_img_path),
        }

    n_found = sum(1 for v in index.values() if v["exists"])
    print(f"  {split}: {len(index)} videos | {n_found} found on disk | "
          f"{len(index)-n_found} missing")
    return index

print("Building video index...")
train_index = build_video_index(train_data, "train") if train_data else {}
val_index   = build_video_index(val_data,   "valid") if val_data   else {}
video_index = {**train_index, **val_index}

# Save locally then cp && sync to Drive
os.makedirs("/content/staging/ytvis", exist_ok=True)
local_index = "/content/staging/ytvis/video_index.json"
with open(local_index, "w") as f:
    json.dump({str(k): v for k, v in video_index.items()}, f, indent=2)

index_path = cp_sync(local_index, PROC_YTVIS)
# Immediate read-back to confirm
with open(index_path) as f:
    n_entries = len(json.load(f))
print(f"\n✅ video_index.json saved: {index_path}")
print(f"   Entries confirmed on Drive: {n_entries} videos")

Building video index...
  train: 2985 videos | 2985 found on disk | 0 missing
  valid: 492 videos | 492 found on disk | 0 missing

✅ video_index.json saved: /content/drive/MyDrive/DLCV_OV_Analytics/datasets/processed/youtube_vis/video_index.json
   Entries confirmed on Drive: 2985 videos


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 4: Convert labels locally, save category_map with
# cp&&sync, labels with tar transfer.
#
# Smart recovery-aware: checks what already exists on Drive
# and skips train labels if recovery script already wrote them.
# ─────────────────────────────────────────────────────────────
import subprocess

def cp_sync(src, dst_dir):
    result = subprocess.run(f"cp '{src}' '{dst_dir}/' && sync",
                            shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        raise IOError(f"cp failed: {result.stderr}")
    return os.path.join(dst_dir, os.path.basename(src))

def convert_ytvis_to_yolo_labels(data, video_index, lbl_out_root,
                                  cat_id_to_yolo, split_name):
    vid_info      = {v["id"]: v for v in data.get("videos", [])}
    stats         = {"n_videos_written": 0, "n_frames_written": 0,
                     "n_boxes": 0, "n_skipped_none": 0, "n_skipped_degen": 0,
                     "class_counts": defaultdict(int)}
    anns_by_video = defaultdict(list)
    for ann in data.get("annotations", []):
        anns_by_video[ann["video_id"]].append(ann)

    for vid_id, anns in tqdm(anns_by_video.items(), desc=split_name):
        if vid_id not in vid_info:
            continue
        video  = vid_info[vid_id]
        length = video["length"]
        fnames = video.get("file_names", [])
        if not fnames:
            continue
        vid_folder  = Path(fnames[0]).parent.name
        img_w, img_h = 1280, 720
        vid_lbl_dir = os.path.join(lbl_out_root, vid_folder)
        os.makedirs(vid_lbl_dir, exist_ok=True)
        frame_lines = defaultdict(list)

        for ann in anns:
            bboxes   = ann.get("bboxes", [])
            yolo_cls = cat_id_to_yolo.get(ann["category_id"])
            if yolo_cls is None:
                continue
            for fi, bbox in enumerate(bboxes):
                if fi >= length:
                    break
                if bbox is None:
                    stats["n_skipped_none"] += 1
                    continue
                x, y, w, h = bbox
                if w <= 0 or h <= 0:
                    stats["n_skipped_degen"] += 1
                    continue
                x = max(0.0, min(x, img_w-1))
                y = max(0.0, min(y, img_h-1))
                w = min(w, img_w-x)
                h = min(h, img_h-y)
                if w <= 0 or h <= 0:
                    stats["n_skipped_degen"] += 1
                    continue
                xc = max(0.0, min(1.0, (x+w/2)/img_w))
                yc = max(0.0, min(1.0, (y+h/2)/img_h))
                wn = max(0.0, min(1.0, w/img_w))
                hn = max(0.0, min(1.0, h/img_h))
                frame_lines[fi].append(
                    f"{yolo_cls} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}"
                )
                stats["n_boxes"] += 1
                stats["class_counts"][yolo_cls] += 1

        for fi in range(length):
            fname = Path(fnames[fi]).stem if fi < len(fnames) else f"{fi:05d}"
            with open(os.path.join(vid_lbl_dir, fname+".txt"), "w") as f:
                f.write("\n".join(frame_lines.get(fi, [])))
            stats["n_frames_written"] += 1
        stats["n_videos_written"] += 1
    return stats


# ── category_map.json — cp && sync ────────────────────────────
ytvis_cat_to_yolo = {cat_id: idx for idx, cat_id in enumerate(sorted(YTVIS_CATS.keys()))}
yolo_to_cat_name  = {idx: YTVIS_CATS[cat_id] for cat_id, idx in ytvis_cat_to_yolo.items()}

os.makedirs("/content/staging/ytvis", exist_ok=True)
local_cat_map = "/content/staging/ytvis/category_map.json"
with open(local_cat_map, "w") as f:
    json.dump({"ytvis_cat_to_yolo": {str(k): v for k, v in ytvis_cat_to_yolo.items()},
               "yolo_to_cat_name" : {str(k): v for k, v in yolo_to_cat_name.items()},
               "n_classes": len(ytvis_cat_to_yolo)}, f, indent=2)
cat_map_path = cp_sync(local_cat_map, PROC_YTVIS)
print(f"✅ category_map.json saved and synced: {cat_map_path}")


# ── TRAIN LABELS: check if recovery already wrote them ────────
LOCAL_YV_STAGE     = "/content/staging/ytvis"
LOCAL_YV_LBL_TRAIN = f"{LOCAL_YV_STAGE}/labels/train"
LOCAL_YV_LBL_VAL   = f"{LOCAL_YV_STAGE}/labels/val"
os.makedirs(LOCAL_YV_LBL_TRAIN, exist_ok=True)
os.makedirs(LOCAL_YV_LBL_VAL,   exist_ok=True)

DRIVE_LBL_TRAIN = os.path.join(PROC_YTVIS, "labels", "train")
n_train_on_drive = len(os.listdir(DRIVE_LBL_TRAIN)) if os.path.isdir(DRIVE_LBL_TRAIN) else 0
EXPECTED_TRAIN_VIDEOS = 2985

if n_train_on_drive >= EXPECTED_TRAIN_VIDEOS:
    # Recovery already wrote all train labels — skip conversion, just collect stats
    print(f"\n✅ Train labels already on Drive ({n_train_on_drive} folders) — skipping re-conversion.")
    print("   Collecting stats from existing Drive labels for Cell 5...")

    # Build train_ytvis_stats from what's already on Drive
    # We re-run conversion to LOCAL only (no Drive write needed) to get stats
    print("   Running convert (local only, for stats)...")
    train_ytvis_stats = convert_ytvis_to_yolo_labels(
        train_data, video_index, LOCAL_YV_LBL_TRAIN,
        ytvis_cat_to_yolo, "train"
    )
    print(f"   Stats collected: {train_ytvis_stats['n_boxes']:,} boxes across "
          f"{train_ytvis_stats['n_videos_written']} videos")
    print("   (Labels NOT re-copied to Drive — existing Drive labels kept as-is)")

else:
    # Train labels incomplete or missing — do full conversion + tar copy
    print(f"\n⚠️  Train labels on Drive: {n_train_on_drive} / {EXPECTED_TRAIN_VIDEOS}")
    print("   Running full train conversion + Drive copy...")
    train_ytvis_stats = convert_ytvis_to_yolo_labels(
        train_data, video_index, LOCAL_YV_LBL_TRAIN,
        ytvis_cat_to_yolo, "train"
    ) if train_data else None

    if train_ytvis_stats:
        print(f"  ✅ Videos: {train_ytvis_stats['n_videos_written']} | "
              f"Frames: {train_ytvis_stats['n_frames_written']:,} | "
              f"Boxes: {train_ytvis_stats['n_boxes']:,}")

    TAR_TRAIN = "/content/staging/ytvis_train_labels.tar.gz"
    subprocess.run(f"tar -czf {TAR_TRAIN} -C {LOCAL_YV_STAGE} labels/train/",
                   shell=True, check=True)
    subprocess.run(f"cp '{TAR_TRAIN}' '{PROC_YTVIS}/' && sync",
                   shell=True, check=True)
    subprocess.run(f"tar -xzf '{PROC_YTVIS}/ytvis_train_labels.tar.gz' -C '{PROC_YTVIS}'",
                   shell=True, check=True)
    subprocess.run(f"rm '{PROC_YTVIS}/ytvis_train_labels.tar.gz' && sync",
                   shell=True, check=True)
    print("✅ Train labels copied to Drive and synced.")


# ── VAL LABELS: recovery never wrote these — always needed ────
DRIVE_LBL_VAL = os.path.join(PROC_YTVIS, "labels", "val")
n_val_on_drive = len(os.listdir(DRIVE_LBL_VAL)) if os.path.isdir(DRIVE_LBL_VAL) else 0
EXPECTED_VAL_VIDEOS = 492

if n_val_on_drive >= EXPECTED_VAL_VIDEOS:
    print(f"\n✅ Val labels already on Drive ({n_val_on_drive} folders) — skipping.")
    val_ytvis_stats = None
else:
    print(f"\nProcessing val labels ({n_val_on_drive} folders found, need {EXPECTED_VAL_VIDEOS})...")
    if val_data and len(val_data.get("annotations", [])) > 0:
        val_ytvis_stats = convert_ytvis_to_yolo_labels(
            val_data, video_index, LOCAL_YV_LBL_VAL, ytvis_cat_to_yolo, "valid"
        )
    else:
        n_created = 0
        for video in (val_data.get("videos", []) if val_data else []):
            fnames = video.get("file_names", [])
            if not fnames:
                continue
            vid_folder = Path(fnames[0]).parent.name
            vid_dir    = os.path.join(LOCAL_YV_LBL_VAL, vid_folder)
            os.makedirs(vid_dir, exist_ok=True)
            for fname in fnames:
                open(os.path.join(vid_dir, Path(fname).stem+".txt"), "w").close()
                n_created += 1
        print(f"  ✅ {n_created:,} empty val label files created locally.")
        val_ytvis_stats = None

    TAR_VAL = "/content/staging/ytvis_val_labels.tar.gz"
    subprocess.run(f"tar -czf {TAR_VAL} -C {LOCAL_YV_STAGE} labels/val/",
                   shell=True, check=True)
    subprocess.run(f"cp '{TAR_VAL}' '{PROC_YTVIS}/' && sync",
                   shell=True, check=True)
    subprocess.run(f"tar -xzf '{PROC_YTVIS}/ytvis_val_labels.tar.gz' -C '{PROC_YTVIS}'",
                   shell=True, check=True)
    subprocess.run(f"rm '{PROC_YTVIS}/ytvis_val_labels.tar.gz' && sync",
                   shell=True, check=True)
    print("✅ Val labels copied to Drive and synced.")


print("\n✅ Cell 4 complete.")

✅ category_map.json saved and synced: /content/drive/MyDrive/DLCV_OV_Analytics/datasets/processed/youtube_vis/category_map.json

✅ Train labels already on Drive (2985 folders) — skipping re-conversion.
   Running convert (local only, for stats)...


train: 100%|██████████| 2985/2985 [00:08<00:00, 360.96it/s]

   Stats collected: 175,384 boxes across 2985 videos
   (Labels NOT re-copied to Drive — existing Drive labels kept as-is)

✅ Val labels already on Drive (492 folders) — skipping.

✅ Cell 4 complete.


In [ ]:
# ─────────────────────────────────────────────────────────────
# CELL 5: Save ytvis_stats.csv + verify everything on Drive
# ─────────────────────────────────────────────────────────────
import pandas as pd

# ── ytvis_stats.csv ───────────────────────────────────────────
rows = []
if train_ytvis_stats:
    for cls_id, count in train_ytvis_stats["class_counts"].items():
        rows.append({"split": "train", "class_id": cls_id,
                     "class_name": yolo_to_cat_name.get(cls_id, f"cls_{cls_id}"),
                     "count": count})

if rows:
    local_stats = "/content/staging/ytvis/ytvis_stats.csv"
    pd.DataFrame(rows).to_csv(local_stats, index=False)
    stats_path = cp_sync(local_stats, PROC_YTVIS)
    print(f"✅ ytvis_stats.csv saved and synced: {stats_path}")
    top5 = pd.DataFrame(rows).groupby("class_name")["count"].sum().nlargest(5)
    print("\nTop 5 classes:"); print(top5.to_string())

# ── Verify everything on Drive ────────────────────────────────
print("\n" + "=" * 55)
print("  YOUTUBE-VIS PREPROCESSING — DRIVE VERIFICATION")
print("=" * 55)

checks = [
    (os.path.join(PROC_YTVIS, "video_index.json"),  "video_index.json"),
    (os.path.join(PROC_YTVIS, "category_map.json"), "category_map.json"),
    (os.path.join(PROC_YTVIS, "ytvis_stats.csv"),   "ytvis_stats.csv"),
    (os.path.join(PROC_YTVIS, "labels", "train"),   "labels/train/ folder"),
    (os.path.join(PROC_YTVIS, "labels", "val"),     "labels/val/ folder"),
]
all_ok = True
for path, name in checks:
    if os.path.isfile(path):
        print(f"  ✅ {name}: {os.path.getsize(path)/1024:.1f} KB")
    elif os.path.isdir(path):
        n = len(os.listdir(path))
        print(f"  ✅ {name}: {n:,} entries")
    else:
        print(f"  ❌ MISSING: {name}")
        all_ok = False

subprocess.run("sync", shell=True)

if all_ok:
    print("\n✅ Milestone 1 COMPLETE. All files confirmed on Drive.")
    print("   Next → Notebook 02_baselines.ipynb")
else:
    print("\n❌ Some files missing — re-run Cell 4 or Cell 5.")

✅ ytvis_stats.csv saved and synced: /content/drive/MyDrive/DLCV_OV_Analytics/datasets/processed/youtube_vis/ytvis_stats.csv

Top 5 classes:
class_name
person         61546
monkey          9269
dog             7531
car             7024
giant_panda     6759

  YOUTUBE-VIS PREPROCESSING — DRIVE VERIFICATION
  ✅ video_index.json: 3755.1 KB
  ✅ category_map.json: 1.4 KB
  ✅ ytvis_stats.csv: 0.8 KB
  ✅ labels/train/ folder: 2,985 entries
  ✅ labels/val/ folder: 492 entries

✅ Milestone 1 COMPLETE. All files confirmed on Drive.
   Next → Notebook 02_baselines.ipynb
